In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Adapted from https://github.com/johnnynunez/blogs/tree/main/Article_2
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md for attribution.


# Simulating a Robot Arm with MuJoCo


Before scaling a simulation, build a task whose state and behavior you can inspect. This lesson introduces [MuJoCo](https://mujoco.readthedocs.io/), a physics engine for robotics simulation.

The robot is the [SO-101](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotstudio_so101), a low-cost arm with five arm joints and a gripper from MuJoCo Menagerie. The task is to grasp a red cube and stack it on a blue cube using contact and friction, without attaching the cube to the gripper or teleporting it during the task.

This lesson runs one world on the CPU using MuJoCo's double-precision state. It establishes a reference task to inspect when moving to MuJoCo Warp on the GPU; matching the task definition does not guarantee identical trajectories across implementations.


## Outline

- Introduce `MjModel`, `MjData`, and `mj_step`.
- Inspect the SO-101, table, and two cubes.
- Step physics and explore forward and inverse kinematics.
- Identify two common grasping problems and the controller choices used here.

The next lesson assembles the full CPU task. The GPU lesson then introduces batched state and measures physics throughput with MuJoCo Warp.


## What you will learn

- Explain the roles of `MjModel`, `MjData`, and `mj_step`.
- Load an MJCF scene, inspect its degrees of freedom, and advance it in time.
- Distinguish forward from inverse kinematics and use both from Python.
- Distinguish single-world latency from aggregate simulation throughput.


## Setup

In a local checkout, install the packages using the [tutorial README](../README.md) and select that environment as the notebook kernel. The cell below locates the checkout and changes into this lesson's directory so that the exercise scripts and imports resolve correctly.

In Google Colab, the cell downloads this tutorial from the repository's `main` branch and installs its pinned dependencies. For the GPU lesson, select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU) first. If Colab requests a runtime restart after installation, restart and rerun the setup cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

lesson_relative = Path("tutorials/sim2real-blogs/notebooks/mujoco/part1")
repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / lesson_relative / "pick_place_common.py").is_file()),
    None,
)
if repo_root is None and in_colab:
    repo_root = Path("/content/accelerated-computing-hub")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "--branch", "main",
             "https://github.com/NVIDIA/accelerated-computing-hub.git",
             str(repo_root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_root), "sparse-checkout", "set",
             "tutorials/sim2real-blogs/notebooks/mujoco"],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
if repo_root is None or not (repo_root / lesson_relative / "pick_place_common.py").is_file():
    raise FileNotFoundError("Open this notebook from an accelerated-computing-hub checkout.")

if in_colab:
    # Colab has no desktop display; use its GPU-backed EGL context.
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(repo_root / "tutorials/sim2real-blogs/notebooks/mujoco/requirements.txt")],
        check=True,
    )
lesson_dir = repo_root / lesson_relative
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
print("Lesson directory:", lesson_dir)


## Environment check

The CPU simulation requires `mujoco` and `numpy`. The optional rendering cell also uses `matplotlib` and a working OpenGL context. No GPU is required for this lesson.


In [ ]:
import mujoco
import numpy as np

print("mujoco :", mujoco.__version__)
print("numpy  :", np.__version__)

## The scene

Alongside this notebook are the task helpers, an exercise script, and reference solutions.

| File | Purpose |
|------|---------|
| `robots.py` | Profiles for the SO-101 and Seeed reBot DevArm (`--robot so101` or `--robot rebot`) |
| `pick_place_common.py` | Scene generation, inverse kinematics, and the waypoint controller |
| `utils.py` | Pinned MuJoCo Menagerie asset download and macOS viewer helpers |
| `so101_pick_place.py` | The exercise script completed in the next lesson |
| `solutions/` | The completed script and one snippet per exercise step |

The generated MJCF includes the robot XML and adds a table and two cubes. **The SO-101 demo increases the compiled arm actuator force limits from ±2.94 to ±30 N·m**, leaving the gripper limit unchanged. This simulation override helps the arm reach the prescribed poses; it is not a claim about the physical SO-101's torque capability. The CPU and GPU examples both use this override through `load_pick_place_model()`.


In [ ]:
from pick_place_common import resolve_pick_place_scene
from robots import get_robot

spec = get_robot("so101")
xml_path = resolve_pick_place_scene(spec=spec)
print(xml_path.read_text())


The robot base is at the world origin, reaching along **+X**. The pick and place positions are separated across **Y**.

On first use, the asset helper downloads the selected robot directory from a pinned MuJoCo Menagerie revision. It generates a scene next to a copy of the robot XML so relative mesh paths resolve correctly. An internet connection and `git` are needed for the first download.


In [ ]:
print("Scene ready:", xml_path)


## MjModel and MjData

| Object | Role |
|--------|------|
| `mujoco.MjModel` | Compiled model: geometry, inertia, joint structure, actuator limits, and simulation options |
| `mujoco.MjData` | Mutable state: `qpos`, `qvel`, `ctrl`, contacts, and simulation time |
| `mujoco.mj_step` | Advances `MjData` by one `model.opt.timestep` |

Treat the compiled model as shared configuration during stepping; configure options before simulation. Separating model and state lets many simulations share a model, including MuJoCo CPU rollouts and the batched GPU states introduced later.

`load_pick_place_model()` applies the documented arm force-limit override. Set the timestep to **0.002 s**, matching the completed task's 50 Hz controller and 10 physics substeps per command.


In [ ]:
from pick_place_common import load_pick_place_model

model = load_pick_place_model(xml_path, spec)
model.opt.timestep = 0.002
data = mujoco.MjData(model)

print(f"nq  (generalized coordinates) : {model.nq}")
print(f"nv  (degrees of freedom)      : {model.nv}")
print(f"nu  (actuators)               : {model.nu}")
print(f"nbody                         : {model.nbody}")
print(f"ngeom                         : {model.ngeom}")
print(f"timestep                      : {model.opt.timestep} s")


`nq` is larger than the number of arm joints because each cube carries a **free joint**: 7 numbers (3 for position, 4 for the orientation quaternion) in `qpos`, but only 6 in `qvel`. That mismatch between `nq` and `nv` is normal in MuJoCo and is the reason positions and velocities are indexed separately.

Let's list the actuators, which are what our controller will drive.

In [ ]:
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    lo, hi = model.actuator_ctrlrange[i]
    print(f"{i}: {name:<14} ctrlrange=[{lo:+.3f}, {hi:+.3f}]")

## Stepping physics

Now let's actually simulate. We put the arm in its grasp-ready home pose, drop the cubes onto the table, and step forward in time.

`apply_arm_ctrl()` writes both `data.ctrl` and the matching `data.qpos` and then runs `mj_forward`, so every body pose is immediately valid. This kind of teleporting is only ever acceptable for **initialization** — once the loop starts, state must evolve through `mj_step`.

In [ ]:
from pick_place_common import apply_arm_ctrl, reset_cubes

apply_arm_ctrl(model, data, spec.home_ctrl)
reset_cubes(model, data, spec)

print("t =", data.time)
print("arm qpos :", np.round(data.qpos[:6], 3))

# Let the scene settle under gravity for half a second of simulated time.
n_steps = int(0.5 / model.opt.timestep)
for _ in range(n_steps):
    mujoco.mj_step(model, data)

print(f"\nAfter {n_steps} steps of mj_step:")
print("t =", round(data.time, 4))
print("arm qpos :", np.round(data.qpos[:6], 3))
print("contacts :", data.ncon)


Two things to inspect:

1. `mj_step(model, data)` advances one state. A Python loop over many `MjData` objects is a **sequential CPU baseline**, not the only CPU execution option: [`mujoco.rollout`](https://mujoco.readthedocs.io/en/stable/python.html#rollout) supports multithreaded CPU rollouts.
2. `data.ncon` reports contacts, including the cubes resting on the table. Contacts affect physics cost, so performance depends on the scene and its evolving state.

Time this single world with fixed actuator targets. This measures physics stepping after settling, not controller execution or successful pick-and-place tasks per second.


In [ ]:
import time

n_timed = 2000
t0 = time.perf_counter()
for _ in range(n_timed):
    mujoco.mj_step(model, data)
elapsed = time.perf_counter() - t0

print(f"{n_timed} steps in {elapsed:.3f} s")
print(f"{elapsed / n_timed * 1e6:.1f} microseconds per step")
print(f"{n_timed / elapsed:,.0f} steps/second (single world)")

This number describes the hardware, scene, state, and settings used in this run. MuJoCo CPU often suits interactive workloads that need a quick individual step; batched simulation instead aims to increase the total number of world-steps per second. Measure both latency and throughput for the workload you intend to run.

---

## Forward kinematics and the grasp point

Forward kinematics answers: *given joint angles, where is everything in the world?* Calling `mj_forward` refreshes derived fields such as `data.xpos`, `data.geom_xpos`, and `data.site_xpos` for the current state.

The SO-101's `gripperframe` site is not the center of the jaw aperture. Using that site as the grasp target can offset the gripper from the cube. The helper instead measures the aperture center between the fixed and moving jaw tips. Inspect the offset below; it depends on the gripper configuration.


In [ ]:
from pick_place_common import aperture_center

mujoco.mj_forward(model, data)

site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "gripperframe")
site_pos = data.site_xpos[site_id]
grasp_pos = aperture_center(model, data)

print("gripperframe site :", np.round(site_pos, 4))
print("aperture center   :", np.round(grasp_pos, 4))
print("offset            :", round(float(np.linalg.norm(site_pos - grasp_pos)) * 1000, 1), "mm")

## Inverse kinematics

Inverse kinematics is the reverse question: *given a desired world position, what joint angles get me there?* There is no closed-form answer for a general arm, so we solve it iteratively with **damped least squares**:

```
repeat:
    run FK at the current guess
    err = target - aperture_center
    J   = positional Jacobian of the aperture point   (mujoco.mj_jac)
    dq  = J^T (J J^T + lambda I)^-1 err
    q  += step * dq
    clamp q to joint limits
```

Two design choices in `pick_place_common.ik_to_target()` are worth understanding:

- **The wrist roll is locked.** The SO-101 has 5 arm degrees of freedom, which cannot reach an arbitrary 6-DoF pose. We fix the redundant joint at the Menagerie *pickup* keyframe value so the jaws open horizontally and can pinch the cube's side faces, then solve position-only IK for the remaining four joints.
- **Damping (lambda) keeps the solve stable** near singularities, at the cost of a small steady-state error. That trade is the right one here.

Let's ask the solver to move the grasp point 8 cm above the red cube.

In [ ]:
from pick_place_common import ik_to_target

target = spec.red_cube_pos + np.array([0.0, 0.0, 0.08])
ik_data = mujoco.MjData(model)
apply_arm_ctrl(model, ik_data, spec.home_ctrl)

ctrl = ik_to_target(
    model, ik_data, target, spec.home_ctrl.copy(), gripper=spec.gripper_open
)
apply_arm_ctrl(model, ik_data, ctrl)
reached = aperture_center(model, ik_data)

print("target   :", np.round(target, 4))
print("reached  :", np.round(reached, 4))
print("residual :", round(float(np.linalg.norm(target - reached)) * 1000, 2), "mm")
print("ctrl     :", np.round(ctrl, 3))


A small residual means the kinematic solve approached the target. The solve uses scratch `MjData`, leaving the live simulation state untouched. The GPU demonstration can therefore reuse this CPU controller while changing the dynamics backend.

---

## Closing the gripper gradually

A sudden close command can accelerate the moving jaw into the cube and push it away before a stable friction grasp forms. The `grasp_red` waypoint in `build_auto_sequence()` uses a gradual close to reduce this effect. Success still depends on contact settings, geometry, and the simulated trajectory.


In [ ]:
from pick_place_common import build_auto_sequence

for wp in build_auto_sequence():
    print(f"{wp.name:<12} target={np.round(wp.target, 3)}  gripper={wp.gripper:.2f}  {wp.duration:.1f}s")

## Render a frame

Finally, let's look at the scene. MuJoCo's offscreen `Renderer` needs a working OpenGL context; if this cell fails in your environment, skip it and use the interactive viewer from the terminal instead (covered in the next notebook).

In [ ]:
try:
    import matplotlib.pyplot as plt

    apply_arm_ctrl(model, data, spec.home_ctrl)
    reset_cubes(model, data)

    with mujoco.Renderer(model, height=480, width=640) as renderer:
        camera = mujoco.MjvCamera()
        mujoco.mjv_defaultCamera(camera)
        camera.azimuth, camera.elevation, camera.distance = 140, -25, 1.0
        camera.lookat[:] = [0.35, -0.04, 0.12]
        renderer.update_scene(data, camera)
        frame = renderer.render()

    plt.figure(figsize=(8, 6))
    plt.imshow(frame)
    plt.axis("off")
    plt.title("SO-101 at the home pose")
    plt.show()
except Exception as exc:  # noqa: BLE001 - rendering is environment dependent
    print("Offscreen rendering unavailable in this environment:")
    print(f"  {type(exc).__name__}: {exc}")
    print("\nUse the interactive viewer instead (from a terminal):")
    print("  python solutions/so101_pick_place_solution.py")


## Recap

- What is the difference between `MjModel` and `MjData`, and why does the split matter for batching?
- Why is `nq` larger than `nv` in this scene?
- Why do we drive inverse kinematics to the aperture center?
- Why does the grasp waypoint close the gripper gradually?
- Which model override prevents this example from representing stock SO-101 actuator strength?

---

## Next

Continue to [Build the Pick and Place](02__pick_and_place.ipynb), where you complete `so101_pick_place.py` and run the full task.
